## 4. LSTM Baseline

In [1]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive

    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

IN_COLAB=False  REPO_ROOT=/Users/ucabtt4/Projects/Dissertation/dissertation


In [2]:
import json

from src.dataloader import load_cgm_stats, load_tensors
from src import metrics
from src import train
from src.models.nn_baseline import LSTMPredictor, HORIZONS

MODEL_NAME = 'LSTM'
SEED = 42  # change this and re-run for a multi-seed stability comparison
# Built from REPO_ROOT, not the current working directory: Colab's cwd is /content regardless of
# where this notebook file itself lives on the mounted Drive, so a plain relative path like
# Path('results') would silently write to the wrong (ephemeral) place.
EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '1_baseline'
RESULTS_PATH = EXPERIMENT_DIR / 'results' / f'lstm_seed{SEED}_results.json'
CHECKPOINT_DIR = EXPERIMENT_DIR / 'checkpoints' / f'lstm_seed{SEED}'

In [3]:
stats = load_cgm_stats()
train_loader = load_tensors('train')  # batch_size=256 default (Training Procedure)
val_loader = load_tensors('val')
test_loader = load_tensors('test')

print('n_train_batches', len(train_loader), 'n_val_batches', len(val_loader),
      'n_test_batches', len(test_loader))

n_train_batches 327 n_val_batches 66 n_test_batches 66


In [4]:
train.set_seed(SEED)  # before constructing the model -- weight init draws from this
model = LSTMPredictor()

model, history = train.train_model(
    model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
    checkpoint_dir=CHECKPOINT_DIR,
)

device: cpu  model: LSTMPredictor  params: 207,749
epoch   1/50  train_loss=0.4756  val_loss=0.3580  val_rmse(mg/dL)=[11.8, 18.3, 25.9, 30.4, 33.3]
epoch   2/50  train_loss=0.4059  val_loss=0.3389  val_rmse(mg/dL)=[11.1, 17.8, 25.3, 29.7, 32.3]
epoch   3/50  train_loss=0.3944  val_loss=0.3401  val_rmse(mg/dL)=[11.1, 17.8, 25.4, 29.8, 32.3]
epoch   4/50  train_loss=0.3901  val_loss=0.3403  val_rmse(mg/dL)=[11.5, 17.9, 25.3, 29.7, 32.3]
epoch   5/50  train_loss=0.3889  val_loss=0.3485  val_rmse(mg/dL)=[11.9, 18.3, 25.7, 30.0, 32.6]
epoch   6/50  train_loss=0.3878  val_loss=0.3426  val_rmse(mg/dL)=[11.3, 17.9, 25.5, 29.9, 32.4]
epoch   7/50  train_loss=0.3850  val_loss=0.3459  val_rmse(mg/dL)=[11.5, 18.0, 25.5, 30.0, 32.6]
epoch   8/50  train_loss=0.3853  val_loss=0.3418  val_rmse(mg/dL)=[11.2, 17.8, 25.3, 29.8, 32.6]
epoch   9/50  train_loss=0.3806  val_loss=0.3373  val_rmse(mg/dL)=[11.3, 17.8, 25.2, 29.6, 32.2]
epoch  10/50  train_loss=0.3787  val_loss=0.3338  val_rmse(mg/dL)=[11.0, 17.

In [ ]:
y_pred_val, y_true_val = train.predict(model, val_loader)
y_pred_test, y_true_test = train.predict(model, test_loader)

val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])

print('=== Validation (best checkpoint) ===')
print(metrics.format_results_table(val_results))
print()
print('=== Test ===')
print(metrics.format_results_table(test_results))

In [ ]:
RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
with open(RESULTS_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME,
        'seed': SEED,
        'val_results': val_results,
        'test_results': test_results,
        'n_epochs_trained': len(history['train_loss']),
    }, f, indent=2)

print(f'saved to {RESULTS_PATH}')
print(f'checkpoints (best.pt / final.pt / history.json) saved to {CHECKPOINT_DIR}')